# MDST 3D Generative Pipeline — Week 5: Text → Parameters

## Goal

Train a small model that reads a plain-English description and predicts the
generator's 10 parameters:

> *"a city in the hills with three separate business hubs and lots of parks"*
> → `terrain=hills`, `city_archetype=polycentric`, `num_clusters≈3`, `green_amount` high, …

By the end you'll have written five files in `text2city/` and trained and
evaluated your own model on your own laptop. Wiring it into
`generate_city.py` (type a sentence, get a city) is **Week 7**.

### The pipeline you're building

```
generate_dataset.py  →  data/text2city_dataset.csv   synthetic sentences + their labels
dataset.py           →  arrays the model can train on
model.py             →  frozen sentence embedding → small trunk → one head per parameter
train.py             →  text2city/checkpoint/
evaluate.py          →  how good is it, really?
```

### How this notebook works (same as Week 3)

- **Run it top to bottom first, before writing anything.** Nothing crashes
  because a file doesn't exist yet — checks print `[PASS]` / `[FAIL]` with a hint.
- Each file is written by a `%%writefile` cell. Fill its `# <-- YOUR CODE HERE`
  lines, run the cell to save the file, then run the check under it.
- **Keep every function name and argument as given** — the files call each other.
- 10 TODOs, each a line or three. Training takes about a minute.


## Part 0 — Setup

Run these in order. They don't depend on anything you haven't written yet.

In [22]:
import os, sys, subprocess, importlib, platform
from pathlib import Path

_here = Path.cwd().resolve()
for _c in [_here, *_here.parents]:
    if (_c / "citygen" / "city_grid.py").exists():
        os.chdir(_c); break
else:
    raise RuntimeError("Couldn't find the repo root (folder containing citygen/city_grid.py). "
                       "Put this notebook inside your project folder and restart the kernel.")

REPO = Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
(REPO / "text2city").mkdir(exist_ok=True)
(REPO / "data").mkdir(exist_ok=True)
print("repo root:", REPO)

repo root: /Users/jemueloppong/Desktop/MDST


In [23]:
def check(label, condition, hint=""):
    """Reports PASS/FAIL. Never raises -- a failure should never stop you."""
    print(f"[{'PASS' if condition else 'FAIL'}] {label}")
    if not condition and hint:
        for line in hint.strip().splitlines():
            print(f"       {line}")
    return bool(condition)


def fresh(module_name):
    """
    Import a text2city module from disk, throwing away any copy Python
    cached earlier -- so a check always sees the file you JUST saved, not an
    older one. Returns None (and says why) if the file won't import.
    """
    for name in list(sys.modules):
        if name == "text2city" or name.startswith("text2city."):
            del sys.modules[name]
    try:
        return importlib.import_module(module_name)
    except Exception as e:
        print(f"[FAIL] couldn't import {module_name}: {type(e).__name__}: {e}")
        return None


def run(args, timeout=1800):
    """Run a command from the repo root, printing its output live. Returns (ok, output)."""
    print("$", " ".join(str(a) for a in args), "\n")
    p = subprocess.Popen([str(a) for a in args], cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    p.wait(timeout=timeout)
    out = "".join(lines)
    return (p.returncode == 0 and "Traceback" not in out), out

print("helpers ready")

helpers ready


### Install the ML packages

This installs into **this notebook's own Python**, so it works whichever
venv you started Jupyter from. On an **Intel Mac** it installs pinned
versions automatically — PyTorch stopped at 2.2.2 for Intel Macs, and newer
packages refuse to work with it. Takes a few minutes the first time.

In [24]:
INTEL_MAC = platform.system() == "Darwin" and platform.machine() == "x86_64"

if INTEL_MAC:
    print("Intel Mac detected -> installing pinned versions")
    steps = [
        ["install", "torch", "scikit-learn", "numpy<2"],
        ["uninstall", "-y", "sentence-transformers", "transformers", "huggingface-hub", "tokenizers", "safetensors"],
        ["install", "sentence-transformers==2.2.2", "transformers==4.30.2", "huggingface-hub==0.16.4"],
    ]
else:
    print(f"{platform.system()} {platform.machine()} -> installing current versions")
    steps = [["install", "torch", "scikit-learn", "sentence-transformers"]]

for step in steps:
    r = subprocess.run([sys.executable, "-m", "pip", "-q", *step], capture_output=True, text=True)
    print(("ok   " if r.returncode == 0 else "FAIL ") + " ".join(step))
    if r.returncode != 0:
        print(r.stderr[-1500:])

if "numpy" in sys.modules:
    print("\nnumpy was already loaded in this kernel -> Kernel > Restart, then run Part 0 again.")

Darwin arm64 -> installing current versions
ok   install torch scikit-learn sentence-transformers

numpy was already loaded in this kernel -> Kernel > Restart, then run Part 0 again.


In [25]:
# Does torch actually work with your numpy? (Catches the Intel Mac ABI problem.)
ok, out = run([sys.executable, "-c",
    "import torch, numpy; torch.tensor(numpy.zeros(2, dtype='float32')); "
    "print('torch', torch.__version__, '| numpy', numpy.__version__)"])
check("torch and numpy work together", ok,
      "If you see 'Could not infer dtype' or '_ARRAY_API not found': pip install \"numpy<2\", "
      "then restart the kernel.")

$ /usr/local/bin/python3 -c import torch, numpy; torch.tensor(numpy.zeros(2, dtype='float32')); print('torch', torch.__version__, '| numpy', numpy.__version__) 

torch 2.15.0 | numpy 2.5.3
[PASS] torch and numpy work together


True

### The sentence embedding model

The first run downloads `all-MiniLM-L6-v2` (~80 MB, needs internet). After
that it's cached and works offline.

In [26]:
ok, out = run([sys.executable, "-c",
    "from sentence_transformers import SentenceTransformer; "
    "m = SentenceTransformer('all-MiniLM-L6-v2'); "
    "print('embedding shape:', m.encode(['a city by the sea']).shape)"])
check("sentence-transformer loads and embeds a sentence", ok and "(1, 384)" in out,
      "Read the error above. On an Intel Mac, 'Disabling PyTorch' means the pins didn't apply --\n"
      "re-run the install cell, restart the kernel, try again. If you're truly stuck, ask an\n"
      "instructor before switching EMBEDDER below.")

$ /usr/local/bin/python3 -c from sentence_transformers import SentenceTransformer; m = SentenceTransformer('all-MiniLM-L6-v2'); print('embedding shape:', m.encode(['a city by the sea']).shape) 

W1101 13:02:08.641000 25267 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8622.85it/s]
embedding shape: (1, 384)
[PASS] sentence-transformer loads and embeds a sentence


True

In [27]:
# Which embedder train.py uses. Leave this alone unless an instructor tells you
# to switch -- "tfidf" needs no download but understands far less English.
EMBEDDER = "sentence-transformer"
print("EMBEDDER =", EMBEDDER)

EMBEDDER = sentence-transformer


In [28]:
# Your Week 2 citygen decides which terrain and city types the model can learn.
ok, out = run([sys.executable, "-c",
    "from citygen import TERRAIN_ARCHETYPES, CITY_ARCHETYPES; "
    "print('terrains:', TERRAIN_ARCHETYPES); print('cities:  ', CITY_ARCHETYPES)"])
check("citygen imports (your Week 2 terrain.py and buildings.py)", ok,
      "The model reads its terrain and city types from your citygen, so it must import cleanly.\n"
      "Run the Week 4 checklist Step 1 to find the problem.")

$ /usr/local/bin/python3 -c from citygen import TERRAIN_ARCHETYPES, CITY_ARCHETYPES; print('terrains:', TERRAIN_ARCHETYPES); print('cities:  ', CITY_ARCHETYPES) 

terrains: ('flat', 'hills')
cities:   ('monocentric', 'polycentric', 'linear')
[PASS] citygen imports (your Week 2 terrain.py and buildings.py)


True

---

## Part 1 — The contract: `schema.py` (given)

Like `city_grid.py` in Week 2, this file is the contract every other file
reads. **Read it, don't edit it.** Three things to notice:

1. **10 parameters, 3 kinds.** *Categorical* (pick one of a set), *continuous*
   (a number in a range), *binary* (true/false). Each kind gets a different
   output head and a different loss later.
2. **`default`** is what the model should predict when a sentence doesn't
   mention a parameter at all.
3. **`mention_prob`** is how often a synthetic sentence mentions that
   parameter. Terrain is mentioned 80% of the time, green override only 10%.

Terrain choices come from **your** citygen. If your `terrain.py` only has
`flat` and `hills`, the model only learns those two, so it can never hand
the generator a terrain type your code doesn't have.

In [29]:
%%writefile text2city/__init__.py
"""text2city: plain-English description -> citygen parameters."""


Overwriting text2city/__init__.py


In [30]:
%%writefile text2city/schema.py
"""
Single source of truth for the 10 city-generator parameters the
text-to-parameter model predicts.

Every other piece (dataset generation, model, training, evaluation, and the
inference script in a later week) imports PARAM_SCHEMA from here instead of
hardcoding parameter names or ranges. If the generator's parameters ever
change, this is the one file to update.

Each parameter is one of three kinds:
    categorical -- a fixed set of named choices (e.g. terrain archetype)
    continuous  -- a float in [low, high]
    binary      -- True/False

`default` is what the model should predict when a parameter is NOT
mentioned in the text at all (see generate_dataset.py). That keeps the
model predictable on short, partial prompts instead of guessing wildly
for anything the user didn't specify.

Terrain choices come from YOUR citygen package, not a fixed list: if your
terrain.py only implements flat and hills, the model only ever learns to
predict flat and hills -- so it can never hand the generator a terrain
type your code doesn't have.
"""
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

try:
    from citygen import TERRAIN_ARCHETYPES, CITY_ARCHETYPES  # noqa: E402
except ImportError as exc:
    raise ImportError(
        "text2city needs your citygen package (terrain.py and buildings.py) to "
        "import cleanly, because the model learns exactly the terrain and city "
        f"types you implemented. Original error: {exc}"
    ) from exc


@dataclass
class ParamSpec:
    name: str
    kind: str  # "categorical" | "continuous" | "binary"
    default: object
    choices: Optional[list] = None       # required for "categorical"
    value_range: Optional[tuple] = None  # required for "continuous", (low, high)
    # how often a synthetic training sentence mentions this parameter at all
    mention_prob: float = 0.35


_TERRAINS = list(TERRAIN_ARCHETYPES)
_TERRAIN_DEFAULT = "hills" if "hills" in _TERRAINS else _TERRAINS[0]
_CITIES = list(CITY_ARCHETYPES)
_CITY_DEFAULT = "monocentric" if "monocentric" in _CITIES else _CITIES[0]


PARAM_SCHEMA: list[ParamSpec] = [
    ParamSpec(
        name="terrain", kind="categorical", default=_TERRAIN_DEFAULT,
        choices=_TERRAINS,
        mention_prob=0.8,
    ),
    ParamSpec(
        name="city_archetype", kind="categorical", default=_CITY_DEFAULT,
        choices=_CITIES,
        mention_prob=0.65,
    ),
    ParamSpec(
        name="num_clusters", kind="continuous", default=1.0,
        value_range=(1.0, 6.0),
        mention_prob=0.25,
    ),
    ParamSpec(
        name="density", kind="continuous", default=0.5,
        value_range=(0.0, 1.0),
        mention_prob=0.55,
    ),
    ParamSpec(
        name="cluster_tightness", kind="continuous", default=0.5,
        value_range=(0.0, 1.0),
        mention_prob=0.3,
    ),
    ParamSpec(
        name="relief", kind="continuous", default=1.0,
        value_range=(0.0, 2.0),
        mention_prob=0.3,
    ),
    ParamSpec(
        name="water_level", kind="continuous", default=0.2,
        value_range=(0.0, 0.6),
        mention_prob=0.3,
    ),
    ParamSpec(
        name="green_amount", kind="continuous", default=0.15,
        value_range=(0.0, 1.0),
        mention_prob=0.45,
    ),
    ParamSpec(
        name="green_override", kind="binary", default=False,
        mention_prob=0.1,
    ),
    ParamSpec(
        name="road_continuity", kind="continuous", default=0.7,
        value_range=(0.0, 1.0),
        mention_prob=0.25,
    ),
]

PARAM_NAMES = [p.name for p in PARAM_SCHEMA]
PARAM_BY_NAME = {p.name: p for p in PARAM_SCHEMA}

CATEGORICAL_PARAMS = [p for p in PARAM_SCHEMA if p.kind == "categorical"]
CONTINUOUS_PARAMS = [p for p in PARAM_SCHEMA if p.kind == "continuous"]
BINARY_PARAMS = [p for p in PARAM_SCHEMA if p.kind == "binary"]


def default_vector() -> dict:
    """The full parameter dict when nothing at all is mentioned."""
    return {p.name: p.default for p in PARAM_SCHEMA}


def to_generator_kwargs(params: dict) -> dict:
    """
    Map this schema's parameter names onto generate_city.py's build_city()
    keyword arguments.

    water_level needs special handling: the model always predicts SOME
    number for it, but generate_terrain()'s own default depends on the
    archetype (land-locked ones default to no water at all). So if the
    prediction sits near the schema default -- meaning the text probably
    said nothing about water -- pass None and let generate_terrain() pick
    the right per-archetype default. Otherwise an unmentioned water level
    can flood a third of a mountain map.
    """
    water_level = params["water_level"]
    water_spec = PARAM_BY_NAME["water_level"]
    water_level_kwarg = None if abs(water_level - water_spec.default) < 0.03 else water_level

    return {
        "terrain_archetype": params["terrain"],
        "city_archetype": params["city_archetype"],
        "clusters": int(round(params["num_clusters"])),
        "density": params["density"],
        "cluster_tightness": params["cluster_tightness"],
        "relief": params["relief"],
        "water_level": water_level_kwarg,
        "green_amount": params["green_amount"],
        "green_override": bool(params["green_override"]),
        "road_continuity": params["road_continuity"],
    }


Overwriting text2city/schema.py


In [31]:
schema = fresh("text2city.schema")
if schema:
    print(f"{'parameter':18s} {'kind':12s} {'default':>12s}  {'mention':>7s}  range / choices")
    for p in schema.PARAM_SCHEMA:
        extra = p.choices if p.kind == "categorical" else (p.value_range or "")
        print(f"{p.name:18s} {p.kind:12s} {str(p.default):>12s}  {p.mention_prob:7.0%}  {extra}")

parameter          kind              default  mention  range / choices
terrain            categorical         hills      80%  ['flat', 'hills']
city_archetype     categorical   monocentric      65%  ['monocentric', 'polycentric', 'linear']
num_clusters       continuous            1.0      25%  (1.0, 6.0)
density            continuous            0.5      55%  (0.0, 1.0)
cluster_tightness  continuous            0.5      30%  (0.0, 1.0)
relief             continuous            1.0      30%  (0.0, 2.0)
water_level        continuous            0.2      30%  (0.0, 0.6)
green_amount       continuous           0.15      45%  (0.0, 1.0)
green_override     binary              False      10%  
road_continuity    continuous            0.7      25%  (0.0, 1.0)


### Given plumbing

Run these two. `embedders.py` turns sentences into vectors; `freshness.py` warns you if your checkpoint is older than your code.

In [32]:
%%writefile text2city/embedders.py
"""
Sentence-embedding backends -- turn a list of sentences into a fixed-size
float array. Same interface for both: fit() on the training texts (a no-op
for the pretrained one), then encode() any list of strings.

  "sentence-transformer" (what we use): all-MiniLM-L6-v2, a pretrained
      model that already understands English. It places "surrounded by
      peaks" close to "surrounded by mountains" even though only one of
      them is in our dataset. Needs `sentence-transformers` and a one-time
      ~80MB download (cached, offline afterwards). It is FROZEN: we never
      train it, only the small model on top.

  "tfidf" (offline fallback): classic word counts. Only knows words it saw
      while fitting, so it can't handle a synonym it never met -- but it
      needs no download, which makes it a useful backup.
"""
from __future__ import annotations

import pickle

import numpy as np


class TfidfEmbedder:
    kind = "tfidf"

    def __init__(self, max_features: int = 1200):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
        self._fitted = False

    def fit(self, texts: list[str]):
        self.vectorizer.fit(texts)
        self._fitted = True
        return self

    def encode(self, texts: list[str]) -> np.ndarray:
        if not self._fitted:
            raise RuntimeError("TfidfEmbedder.fit() must be called before encode()")
        return self.vectorizer.transform(texts).toarray().astype(np.float32)

    def save(self, path: str):
        with open(path, "wb") as f:
            pickle.dump(self.vectorizer, f)

    @classmethod
    def load(cls, path: str) -> "TfidfEmbedder":
        obj = cls.__new__(cls)
        with open(path, "rb") as f:
            obj.vectorizer = pickle.load(f)
        obj._fitted = True
        return obj


class SentenceTransformerEmbedder:
    kind = "sentence-transformer"

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)
        self.model_name = model_name

    def fit(self, texts: list[str]):
        return self  # pretrained and frozen -- nothing to fit

    def encode(self, texts: list[str]) -> np.ndarray:
        return np.asarray(self.model.encode(texts, convert_to_numpy=True), dtype=np.float32)

    def save(self, path: str):
        # nothing to save but the name -- the weights live in the download cache
        with open(path, "w") as f:
            f.write(self.model_name)

    @classmethod
    def load(cls, path: str) -> "SentenceTransformerEmbedder":
        with open(path) as f:
            return cls(f.read().strip())


def get_embedder(kind: str, **kwargs):
    if kind == "tfidf":
        return TfidfEmbedder(**kwargs)
    if kind == "sentence-transformer":
        return SentenceTransformerEmbedder(**kwargs)
    raise ValueError(f"unknown embedder kind {kind!r}, choose 'tfidf' or 'sentence-transformer'")


def embedder_filename(kind: str) -> str:
    return "embedder.pkl" if kind == "tfidf" else "embedder.txt"


def load_embedder(kind: str, path: str):
    if kind == "tfidf":
        return TfidfEmbedder.load(path)
    if kind == "sentence-transformer":
        return SentenceTransformerEmbedder.load(path)
    raise ValueError(f"unknown embedder kind {kind!r}")


Overwriting text2city/embedders.py


In [33]:
%%writefile text2city/freshness.py
"""
Staleness detection for checkpoints.

Editing generate_dataset.py, schema.py, dataset.py or model.py does NOTHING
to a checkpoint that's already trained -- you have to re-run
generate_dataset.py and train.py for the change to count. It's easy to
forget, and then you spend an hour asking why your fix doesn't work.

train.py stores a fingerprint of the dataset CSV plus those source files in
the checkpoint. evaluate.py recomputes it and warns loudly if they differ.
"""
from __future__ import annotations

import hashlib
from pathlib import Path

SOURCE_FILES = ["schema.py", "generate_dataset.py", "dataset.py", "model.py"]


def fingerprint_sources(dataset_path: str) -> str:
    hasher = hashlib.sha256()
    dataset_file = Path(dataset_path)
    hasher.update(dataset_file.read_bytes() if dataset_file.exists() else b"MISSING_DATASET")
    here = Path(__file__).resolve().parent
    for name in SOURCE_FILES:
        path = here / name
        if path.exists():
            hasher.update(path.read_bytes())
    return hasher.hexdigest()[:16]


def check_staleness(checkpoint_fingerprint, dataset_path: str) -> None:
    """Prints a loud warning (never raises) if the checkpoint looks stale."""
    if checkpoint_fingerprint is None:
        print("WARNING: checkpoint has no fingerprint (trained by an older train.py) -- consider retraining.")
        return
    if fingerprint_sources(dataset_path) != checkpoint_fingerprint:
        print("=" * 70)
        print("WARNING: this checkpoint appears STALE.")
        print("The dataset CSV and/or schema.py / generate_dataset.py / dataset.py /")
        print("model.py changed since it was trained. Results below reflect the OLD code.")
        print("Fix: re-run generate_dataset.py, then train.py.")
        print("=" * 70)


Overwriting text2city/freshness.py


---

## Part 2 — The dataset: `generate_dataset.py`

No dataset exists that pairs city descriptions with our 10 numbers, so we
**make one**, the same trick as the procedural generator. For each example:

1. For each parameter, roll the dice: mention it or not (`mention_prob`).
2. If mentioned, pick a value **and** a phrase that describes that value —
   `0.92` density goes with *"packed with buildings"*, never with *"sparse"*.
3. If not mentioned, it keeps its **default**.
4. Join the phrases into one sentence.

That's **partial mentions**, and it's the key idea of the week. Real people
describe 2–3 things, not 10. Training on partial sentences teaches the
model to say "default" for whatever wasn't mentioned, instead of guessing.

Given: the phrase banks (long and boring) and the block that keeps
`num_clusters` and `city_archetype` consistent. Read its comment. You write
**TODO 1–3**.

In [34]:
%%writefile text2city/generate_dataset.py
"""
Builds a synthetic (sentence -> parameter vector) dataset to train the
text-to-parameter model on.

Why synthetic: no dataset exists that pairs "city description text" with
these exact 10 generator parameters. So, the same trick as the procedural
generator itself: we write rules that produce (sentence, parameters) pairs
and sample them thousands of times.

Key idea -- PARTIAL mentions, not every combination:
    A real prompt like "a seaside city with a dense center and lots of green
    space" describes ~3 of the 10 parameters. So for each example we pick a
    random SUBSET of parameters to mention (each with its schema
    mention_prob), phrase only those, and leave the rest at their schema
    default. That teaches the model what we actually want: use what's said,
    fall back to sensible defaults for what isn't.

    It's also why we don't need every combination of all 10 parameters:
    each output head learns its own phrase -> value mapping, so a few
    thousand random examples are enough for those mappings to recombine
    freely at prediction time.

    python text2city/generate_dataset.py --n 15000 --seed 0 --out data/text2city_dataset.csv
"""
from __future__ import annotations

import argparse
import csv
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from text2city.schema import PARAM_BY_NAME, PARAM_NAMES, default_vector  # noqa: E402

# ---------------------------------------------------------------------------
# Phrase banks. Every phrase must read naturally after "A city ..." so any
# subset can be joined into one sentence.
# ---------------------------------------------------------------------------

TERRAIN_PHRASES = {
    "flat": ["on flat land", "built on completely flat terrain", "on level ground"],
    "hills": ["set in rolling hills", "spread across gentle hills", "on hilly terrain"],
    "mountains": ["surrounded by mountains", "nestled in rugged mountains", "high in the mountains"],
    "valley": ["nestled in a deep valley", "tucked into a valley", "set in a river valley"],
    "coastal_harbor": ["built around a harbor", "hugging the coastline", "by the sea", "on the coast",
                       "a seaside setting", "along the seaside", "a coastal city", "a coastal setting",
                       "coastal terrain"],
    "peninsula": ["built on a peninsula", "surrounded by water on three sides", "on a narrow peninsula",
                  "a peninsula city", "peninsula terrain"],
    "river": ["split by a winding river", "built along a river", "with a river cutting through it",
              "a river city", "river terrain"],
}


def terrain_phrases(name: str) -> list[str]:
    """Phrases for a terrain type -- with a fallback for a type you invented yourself."""
    return TERRAIN_PHRASES.get(name, [f"with {name.replace('_', ' ')} terrain"])


CITY_ARCHETYPE_PHRASES = {
    "monocentric": ["with a single downtown core", "built around one central business district",
                    "with one clear city center", "a monocentric city", "a monocentric layout",
                    "this city is monocentric"],
    "polycentric": ["with several distinct business districts", "made up of multiple downtown clusters",
                    "with scattered urban centers", "with multiple business hubs",
                    "with two separate downtown areas", "made up of several separate city centers",
                    "a polycentric city", "a polycentric layout", "this city is polycentric"],
    "linear": ["stretched out in a long corridor", "elongated along a single axis", "laid out in a line",
               "long and narrow", "thin and elongated in shape", "a linear city", "a linear layout",
               "this city is linear", "a wide, elongated shape", "a wide and stretched out shape"],
}

# Exact-count phrases, each tied to its own literal number. "one main hub"
# must never be paired with a target of 1.8 just because it shared a range
# with "two hubs" -- the value has to match what the words say.
_CLUSTER_NOUNS = ["hub", "center", "cluster", "district", "node"]
_NUMBER_WORDS = {1: "one", 2: "two", 3: "three", 4: "four", 5: "five", 6: "six"}


def _cluster_noun_phrases(count: int) -> list[str]:
    word = _NUMBER_WORDS[count]
    plural = "" if count == 1 else "s"
    prefix = "main " if count == 1 else ""
    return [f"with {word} {prefix}{noun}{plural}" for noun in _CLUSTER_NOUNS]


NUM_CLUSTERS_EXACT = {
    1: _cluster_noun_phrases(1) + ["with just one main hub", "with a single center"],
    2: _cluster_noun_phrases(2) + ["with a couple of centers", "with just one or two hubs"],
    3: _cluster_noun_phrases(3) + ["with a handful of clusters", "with several separate hubs"],
    4: _cluster_noun_phrases(4) + ["with several separate districts"],
    5: _cluster_noun_phrases(5) + ["with many scattered clusters"],
    6: _cluster_noun_phrases(6) + ["with a large number of distinct hubs"],
}

# Continuous parameters: a list of levels, each ((low, high), [phrases]).
# A phrase from a level is always paired with a value inside that level.
CONTINUOUS_LEVELS = {
    "density": [
        ((0.0, 0.15), ["barely built up", "almost entirely empty", "sparsely inhabited", "sparse", "very sparse"]),
        ((0.15, 0.35), ["lightly built up", "fairly sparse", "not very built up"]),
        ((0.35, 0.6), ["moderately built up", "with medium density", "reasonably built up"]),
        ((0.6, 0.8), ["densely populated", "densely built", "quite crowded with buildings", "dense"]),
        ((0.8, 1.0), ["extremely dense", "packed with buildings", "ultra-dense and crowded", "very dense",
                      "highly concentrated"]),
    ],
    "cluster_tightness": [
        ((0.0, 0.3), ["with buildings spread loosely across the city", "with a sprawling, spread-out layout",
                      "spread out", "sparse layout"]),
        ((0.3, 0.6), ["with a moderately clustered downtown", "with buildings loosely grouped together"]),
        ((0.6, 1.0), ["with a tightly clustered downtown core", "with a compact, concentrated skyline",
                      "concentrated", "compact", "tightly clustered"]),
    ],
    "relief": [
        ((0.0, 0.5), ["with almost no elevation change", "on very flat, gentle ground"]),
        ((0.5, 1.0), ["with gently varying elevation", "with mild rolling terrain"]),
        ((1.2, 2.0), ["with dramatic peaks and valleys", "with extreme, jagged elevation changes"]),
    ],
    "water_level": [
        ((0.0, 0.1), ["with barely any water", "with just a trickle of water"]),
        ((0.1, 0.25), ["with a modest amount of water", "with a small river or bay"]),
        ((0.25, 0.45), ["with a fair amount of water", "with a sizeable harbor"]),
        ((0.45, 0.6), ["with a large body of water", "with an expansive bay"]),
    ],
    "green_amount": [
        ((0.0, 0.15), ["with very few parks", "with almost no greenery"]),
        ((0.15, 0.4), ["with some green spaces", "with a handful of parks"]),
        ((0.4, 0.7), ["with lots of green areas", "with many parks scattered around", "full of parks and trees"]),
        ((0.7, 1.0), ["covered almost entirely in greenery", "mostly parks and forest", "covered in green space",
                      "mostly green space throughout"]),
    ],
    "road_continuity": [
        ((0.0, 0.15), ["with almost no roads at all", "with barely any streets", "with very few roads"]),
        ((0.15, 0.3), ["with a broken, patchy street network", "with disconnected, fragmented roads"]),
        ((0.3, 0.7), ["with a fairly normal street layout", "with an ordinary road network"]),
        ((0.7, 1.0), ["with a fully connected grid of streets", "with a dense, well-connected road network"]),
    ],
}

GREEN_OVERRIDE_PHRASES = [
    "reclaimed almost entirely by nature",
    "overgrown, with greenery taking over the streets",
]

SENTENCE_STARTERS = [
    "A city {items}.",
    "Generate a city {items}.",
    "I'd like a city {items}.",
    "Make me a city {items}.",
    "Design a city {items}.",
]


def _sample_continuous(rng: random.Random, levels: list):
    """Pick one level, then a value inside it and a phrase from it. Returns (value, phrase)."""
    # TODO 1: pick ONE level at random, then a value inside that level's
    # range and one of that level's phrases.
    # Each level looks like ((low, high), [phrase, phrase, ...]), so you can
    # unpack one with:   (low, high), phrases = level
    # Hints: rng.choice(a_list) picks an item; rng.uniform(a, b) picks a float.
    level = rng.choice(levels)
    (low, high), phrases = level
    value = rng.uniform(low, high)
    phrase = rng.choice(phrases)
    return value, phrase


def _join_phrases(phrases: list[str]) -> str:
    """['a', 'b', 'c'] -> 'a, b, and c'   ['a'] -> 'a'"""
    if len(phrases) == 1:
        return phrases[0]
    return ", ".join(phrases[:-1]) + ", and " + phrases[-1]


def generate_example(rng: random.Random) -> dict:
    params = default_vector()
    mentioned_phrases = []

    # --- num_clusters and city_archetype are resolved together, first ---
    # They're the one pair with a real logical link: "five centers" always
    # means a non-monocentric city. Handling them together guarantees the
    # sentence never says "five centers" while the label says monocentric.
    num_clusters_spec = PARAM_BY_NAME["num_clusters"]
    city_spec = PARAM_BY_NAME["city_archetype"]

    num_clusters_mentioned = rng.random() <= num_clusters_spec.mention_prob
    if num_clusters_mentioned:
        count = rng.choice(list(NUM_CLUSTERS_EXACT.keys()))
        mentioned_phrases.append(rng.choice(NUM_CLUSTERS_EXACT[count]))
        params["num_clusters"] = max(1.0, min(6.0, count + rng.uniform(-0.25, 0.25)))
    implies_non_monocentric = num_clusters_mentioned and params["num_clusters"] > 1.5

    non_mono = [c for c in city_spec.choices if c != "monocentric"]
    if rng.random() <= city_spec.mention_prob:
        choices = non_mono if (implies_non_monocentric and non_mono) else city_spec.choices
        value = rng.choice(choices)
        mentioned_phrases.append(rng.choice(CITY_ARCHETYPE_PHRASES[value]))
        params["city_archetype"] = value
    elif implies_non_monocentric and non_mono:
        params["city_archetype"] = rng.choice(non_mono)

    # --- every other parameter: mention it or leave it at its default ---
    for spec in PARAM_BY_NAME.values():
        if spec.name in ("num_clusters", "city_archetype"):
            continue  # handled above

        # TODO 2: should this sentence mention this parameter? It should with
        # probability spec.mention_prob. If it should NOT, `continue` -- the
        # parameter then keeps its default value, which is the whole point.
        # Hint: rng.random() is a float in [0, 1).
        if rng.random() > spec.mention_prob:
            continue

        if spec.name == "terrain":
            value = rng.choice(spec.choices)
            phrase = rng.choice(terrain_phrases(value))
        elif spec.name == "green_override":
            value = True
            phrase = rng.choice(GREEN_OVERRIDE_PHRASES)
        else:
            value, phrase = _sample_continuous(rng, CONTINUOUS_LEVELS[spec.name])

        # TODO 3: record it -- store `value` in params under this parameter's
        # name, and add `phrase` to mentioned_phrases. Both, or the sentence
        # and its label won't agree.
        params[spec.name] = value
        mentioned_phrases.append(phrase)

    if not mentioned_phrases:
        # guarantee at least one mention so the sentence isn't empty
        (low, high), phrases = CONTINUOUS_LEVELS["density"][2]
        mentioned_phrases.append(rng.choice(phrases))
        params["density"] = rng.uniform(low, high)

    rng.shuffle(mentioned_phrases)
    sentence = rng.choice(SENTENCE_STARTERS).format(items=_join_phrases(mentioned_phrases))
    return {"text": sentence, **params}


def generate_dataset(n: int, seed: int = 0) -> list:
    rng = random.Random(seed)
    return [generate_example(rng) for _ in range(n)]


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--n", type=int, default=15000, help="number of examples to generate")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--out", type=str, default="data/text2city_dataset.csv")
    args = parser.parse_args()

    rows = generate_dataset(args.n, seed=args.seed)

    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["text"] + PARAM_NAMES)
        writer.writeheader()
        writer.writerows(rows)

    print(f"wrote {len(rows)} examples to {out_path}")
    print("\nsample rows:")
    for row in rows[:5]:
        print(f"  {row['text']}")
        print(f"    -> {({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items() if k != 'text'})}")


if __name__ == "__main__":
    main()


Overwriting text2city/generate_dataset.py


In [35]:
import random

gd = fresh("text2city.generate_dataset")
if gd:
    # TODO 1 -- _sample_continuous
    try:
        levels = gd.CONTINUOUS_LEVELS["density"]
        rng = random.Random(0)
        samples = [gd._sample_continuous(rng, levels) for _ in range(500)]
        consistent = all(any(lo <= v <= hi and p in ph for (lo, hi), ph in levels) for v, p in samples)
        check("TODO 1: value and phrase always come from the SAME level", consistent,
              "Pick one level first, then take BOTH the value and the phrase from it.")
        check("TODO 1: different calls pick different levels", len({p for _, p in samples}) > 5,
              "Every call returned the same phrase -- use rng.choice on the list of levels.")
    except Exception as e:
        check("TODO 1 runs", False, f"{type(e).__name__}: {e}")

    # TODO 2 + 3 -- the mention loop
    try:
        rows = gd.generate_dataset(4000, seed=0)
        checked = ["density", "cluster_tightness", "relief", "green_amount", "road_continuity"]
        rates = {}
        for name in checked:
            spec = gd.PARAM_BY_NAME[name]
            rates[name] = sum(abs(r[name] - spec.default) > 1e-9 for r in rows) / len(rows)
        close = all(abs(rates[n] - gd.PARAM_BY_NAME[n].mention_prob) < 0.06 for n in checked)
        check("TODO 2+3: each parameter is mentioned about mention_prob of the time", close,
              "Mention rates: " + ", ".join(f"{n}={rates[n]:.0%} (want ~{gd.PARAM_BY_NAME[n].mention_prob:.0%})"
                                             for n in checked) +
              "\nAll ~0%? TODO 2 is always skipping, or TODO 3 isn't storing the value."
              "\nAll ~100%? TODO 2's comparison is the wrong way round.")

        agree = total = 0
        default = gd.PARAM_BY_NAME["green_amount"].default
        for r in rows:
            v = r["green_amount"]
            if abs(v - default) < 1e-9:
                continue
            total += 1
            agree += any(lo <= v <= hi and any(p in r["text"] for p in ph)
                         for (lo, hi), ph in gd.CONTINUOUS_LEVELS["green_amount"])
        check("TODO 3: each label matches a phrase in its own sentence", total > 0 and agree / total > 0.99,
              "A value was stored without its phrase, or a phrase added without its value.")

        print("\nthree example rows:")
        for r in rows[:3]:
            print(" ", r["text"])
    except Exception as e:
        check("TODO 2+3 run", False, f"{type(e).__name__}: {e}")

[PASS] TODO 1: value and phrase always come from the SAME level
[PASS] TODO 1: different calls pick different levels
[PASS] TODO 2+3: each parameter is mentioned about mention_prob of the time
[PASS] TODO 3: each label matches a phrase in its own sentence

three example rows:
  I'd like a city with medium density, on hilly terrain, and with almost no greenery.
  Generate a city highly concentrated, with gently varying elevation, on level ground, with one clear city center, and with a single center.
  Design a city with barely any water, this city is linear, and on hilly terrain.


When the checks pass, generate the real dataset: 15,000 examples, a second or two.

In [36]:
CSV = REPO / "data" / "text2city_dataset.csv"
ok, out = run([sys.executable, "text2city/generate_dataset.py", "--n", "15000", "--seed", "0", "--out", CSV])
n_rows = sum(1 for _ in open(CSV)) - 1 if CSV.exists() else 0
check("data/text2city_dataset.csv has 15000 rows", ok and n_rows == 15000, f"found {n_rows}")
if CSV.exists():
    import csv as _csv
    _texts = {r["text"] for r in _csv.DictReader(open(CSV))}
    check("the sentences are varied (not all defaults)", len(_texts) > 5000,
          f"only {len(_texts)} different sentences -- go back and finish TODO 1-3, then re-run this cell.")

$ /usr/local/bin/python3 text2city/generate_dataset.py --n 15000 --seed 0 --out /Users/jemueloppong/Desktop/MDST/data/text2city_dataset.csv 

wrote 15000 examples to /Users/jemueloppong/Desktop/MDST/data/text2city_dataset.csv

sample rows:
  I'd like a city with medium density, on hilly terrain, and with almost no greenery.
    -> {'terrain': 'hills', 'city_archetype': 'monocentric', 'num_clusters': 1.0, 'density': 0.59, 'cluster_tightness': 0.5, 'relief': 1.0, 'water_level': 0.2, 'green_amount': 0.09, 'green_override': False, 'road_continuity': 0.7}
  Generate a city highly concentrated, with gently varying elevation, on level ground, with one clear city center, and with a single center.
    -> {'terrain': 'flat', 'city_archetype': 'monocentric', 'num_clusters': 1.09, 'density': 0.85, 'cluster_tightness': 0.5, 'relief': 0.86, 'water_level': 0.2, 'green_amount': 0.15, 'green_override': False, 'road_continuity': 0.7}
  Design a city with barely any water, this city is linear, and on hil

---

## Part 3 — Targets: `dataset.py`

The CSV is all text. A model needs numbers, in the form each head expects:

| kind | CSV says | target | why |
|---|---|---|---|
| categorical | `"hills"` | `1` (its index in `choices`) | cross-entropy wants class indices |
| continuous | `"0.92"` | `0.92` rescaled to 0–1 | the head ends in a sigmoid, which only outputs 0–1 |
| binary | `"True"` | `1.0` | binary cross-entropy wants 0.0 / 1.0 |

You write **TODO 4**.

In [37]:
%%writefile text2city/dataset.py
"""
Loads the synthetic (sentence -> parameters) CSV and turns it into the
arrays the training loop needs:

    categorical -> int64 class index   (which choice, as a number)
    continuous  -> float32 in [0, 1]   (rescaled, to match the model's sigmoid output)
    binary      -> float32 0.0 or 1.0
"""
from __future__ import annotations

import csv
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from text2city.schema import PARAM_SCHEMA  # noqa: E402


def load_rows(csv_path: str) -> list[dict]:
    with open(csv_path, newline="") as f:
        return list(csv.DictReader(f))


def build_targets(rows: list[dict]) -> dict:
    """
    Returns {param_name: np.ndarray of shape (N,)}, one array per parameter.

    For continuous parameters it ALSO returns the original, un-rescaled
    values under "<name>__raw" -- handy for reporting errors in real units
    (density off by 0.04) rather than rescaled ones.

    Everything in the CSV is a string, so convert as you go.
    """
    targets = {}
    for spec in PARAM_SCHEMA:
        col = [r[spec.name] for r in rows]
        if spec.kind == "categorical":
            # TODO 4a: each value is a choice name like "hills". Store its
            # POSITION in spec.choices instead (spec.choices.index(v)), as int64.
            targets[spec.name] = np.array([spec.choices.index(v) for v in col], dtype=np.int64)
        elif spec.kind == "continuous":
            raw = np.array([float(v) for v in col], dtype=np.float32)
            low, high = spec.value_range
            # TODO 4b: rescale raw from [low, high] to [0, 1], as float32.
            # The model's continuous heads end in a sigmoid, which can only
            # output 0-1 -- so the targets have to live there too.
            targets[spec.name] = ((raw - low) / (high - low)).astype(np.float32)
            targets[f"{spec.name}__raw"] = raw
        elif spec.kind == "binary":
            # TODO 4c: the CSV holds the TEXT "True" or "False". Store 1.0 / 0.0, as float32.
            targets[spec.name] = np.array([v == "True" for v in col], dtype=np.float32)
    return targets


def train_val_split(n: int, val_fraction: float = 0.15, seed: int = 0):
    """Shuffle 0..n-1 and cut off the last val_fraction for validation. Returns (train_idx, val_idx)."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_val = int(n * val_fraction)
    return idx[n_val:], idx[:n_val]


Overwriting text2city/dataset.py


In [38]:
import numpy as np

ds = fresh("text2city.dataset")
if ds and CSV.exists():
    rows = ds.load_rows(str(CSV))
    try:
        t = ds.build_targets(rows)
        for spec in ds.PARAM_SCHEMA:
            a = t.get(spec.name)
            if not isinstance(a, np.ndarray):
                check(f"TODO 4: {spec.name} is a numpy array", False, f"got {type(a).__name__}")
                continue
            if spec.kind == "categorical":
                ok = (a.dtype == np.int64 and a.min() >= 0 and a.max() < len(spec.choices)
                      and spec.choices[a[0]] == rows[0][spec.name])
                check(f"TODO 4a: {spec.name} -> int64 class indices", ok,
                      f"dtype={a.dtype}, values {a.min()}..{a.max()}, {len(spec.choices)} choices")
            elif spec.kind == "continuous":
                low, high = spec.value_range
                back = low + a * (high - low)
                ok = (a.dtype == np.float32 and a.min() >= -1e-6 and a.max() <= 1 + 1e-6
                      and np.allclose(back, t[f"{spec.name}__raw"], atol=1e-4))
                check(f"TODO 4b: {spec.name} -> float32 in [0, 1]", ok,
                      f"dtype={a.dtype}, range {a.min():.3f}..{a.max():.3f}. "
                      "Subtract low, divide by (high - low).")
            else:
                truth = np.array([r[spec.name] == "True" for r in rows], dtype=np.float32)
                ok = a.dtype == np.float32 and set(np.unique(a)) <= {0.0, 1.0} and np.array_equal(a, truth)
                check(f"TODO 4c: {spec.name} -> float32 0/1", ok, "Compare the text with \"True\".")
    except Exception as e:
        check("TODO 4 runs", False, f"{type(e).__name__}: {e}")

[PASS] TODO 4a: terrain -> int64 class indices
[PASS] TODO 4a: city_archetype -> int64 class indices
[PASS] TODO 4b: num_clusters -> float32 in [0, 1]
[PASS] TODO 4b: density -> float32 in [0, 1]
[PASS] TODO 4b: cluster_tightness -> float32 in [0, 1]
[PASS] TODO 4b: relief -> float32 in [0, 1]
[PASS] TODO 4b: water_level -> float32 in [0, 1]
[PASS] TODO 4b: green_amount -> float32 in [0, 1]
[PASS] TODO 4c: green_override -> float32 0/1
[PASS] TODO 4b: road_continuity -> float32 in [0, 1]


---

## Part 4 — The model: `model.py`

```
sentence ─► [frozen embedding, 384 numbers] ─► trunk (2 layers) ─┬─► terrain head        (one score per choice)
                                                                 ├─► city_archetype head (one score per choice)
                                                                 ├─► density head         (1 number)
                                                                 └─► …one head per parameter
```

The embedding model is pretrained and **frozen**: it already understands
English, and we never change it. We only train the trunk and heads, a few
tens of thousands of weights, which is why it trains in about a minute.
Every head reads the same trunk features but predicts on its own, so
parameters can vary independently.

You write **TODO 5–6**. `decode_outputs` (raw outputs → real values) is
given; read it.

In [39]:
%%writefile text2city/model.py
"""
Text -> city-parameter model.

    sentence
      -> frozen sentence embedding (embedders.py; NOT trained)
      -> small shared MLP "trunk"            <- trained
      -> one output "head" per parameter     <- trained
           categorical: one logit per choice (softmax / cross-entropy)
           continuous:  one number, squashed to 0-1 by a sigmoid, then
                        rescaled into that parameter's [low, high]
           binary:      one logit (sigmoid > 0.5 means True)

Why this shape: the embedding model already knows English. All we train is
the mapping from "meaning of the sentence" to "this generator's 10
numbers" -- a few tens of thousands of weights, which is why it trains in
about a minute on a laptop CPU.

Why one head per parameter: each head reads the same trunk features but
makes its own prediction, so density and relief are free to vary
independently instead of being forced to move together.
"""
from __future__ import annotations

import torch
import torch.nn as nn

from text2city.schema import PARAM_SCHEMA


class TextToCityParams(nn.Module):
    def __init__(self, embedding_dim: int, hidden_dim: int = 128):
        super().__init__()

        # shared trunk: embedding -> hidden -> hidden
        self.trunk = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # one head per parameter, stored by name so training code can find
        # each head without depending on order
        self.heads = nn.ModuleDict()
        for spec in PARAM_SCHEMA:
            # TODO 5: one nn.Linear(in_features, out_features) per parameter,
            # reading the trunk's hidden_dim features. How many outputs?
            # A categorical head needs one score per choice; a continuous or
            # binary head needs a single number.
            if spec.kind == "categorical":
                self.heads[spec.name] = nn.Linear(hidden_dim, len(spec.choices))
            elif spec.kind in ("continuous", "binary"):
                self.heads[spec.name] = nn.Linear(hidden_dim, 1)
            else:
                raise ValueError(f"unknown parameter kind: {spec.kind}")

    def forward(self, embedding: torch.Tensor) -> dict:
        """
        embedding: (batch, embedding_dim). Returns {param_name: raw output}
        -- logits for categorical/binary, unsquashed values for continuous.
        decode_outputs() turns these into real parameter values.
        """
        # TODO 6: run the embedding through the shared trunk ONCE, then hand
        # those same features to every head. Return {name: head_output}.
        # Hint: self.heads.items() gives you (name, head) pairs.
        features = self.trunk(embedding)
        return {name: head(features) for name, head in self.heads.items()}


def decode_outputs(raw_outputs: dict) -> dict:
    """
    One example's raw outputs (no batch dimension) -> real parameter values:
        categorical -> the name of the highest-scoring choice
        continuous  -> sigmoid(raw), rescaled into [low, high]
        binary      -> sigmoid(raw) > 0.5
    """
    result = {}
    schema_by_name = {p.name: p for p in PARAM_SCHEMA}
    for name, raw in raw_outputs.items():
        spec = schema_by_name[name]
        if spec.kind == "categorical":
            result[name] = spec.choices[torch.argmax(raw).item()]
        elif spec.kind == "continuous":
            low, high = spec.value_range
            result[name] = low + torch.sigmoid(raw).item() * (high - low)
        elif spec.kind == "binary":
            result[name] = torch.sigmoid(raw).item() > 0.5
    return result


def decode_batch(raw_outputs: dict, index: int) -> dict:
    """decode_outputs() for one row of a batched forward() output."""
    return decode_outputs({name: raw[index] for name, raw in raw_outputs.items()})


Overwriting text2city/model.py


In [40]:
import torch

mm = fresh("text2city.model")
if mm:
    try:
        model = mm.TextToCityParams(embedding_dim=384, hidden_dim=64)
        check("TODO 5: model builds", True)
        try:
            out = model(torch.randn(8, 384))
            names = [p.name for p in mm.PARAM_SCHEMA]
            check("TODO 6: forward returns one output per parameter",
                  isinstance(out, dict) and set(out) == set(names),
                  f"got {type(out).__name__} with keys {sorted(out) if isinstance(out, dict) else '-'}")
            if isinstance(out, dict):
                for p in mm.PARAM_SCHEMA:
                    want = (8, len(p.choices)) if p.kind == "categorical" else (8, 1)
                    got = tuple(out[p.name].shape) if p.name in out else None
                    check(f"TODO 5: {p.name} head outputs {want}", got == want, f"got {got}")
            n = sum(x.numel() for x in model.parameters())
            print(f"\ntrainable weights (hidden_dim=64): {n:,}")
        except Exception as e:
            check("TODO 6: forward runs", False, f"{type(e).__name__}: {e}")
    except Exception as e:
        check("TODO 5: model builds", False, f"{type(e).__name__}: {e}")

[PASS] TODO 5: model builds
[PASS] TODO 6: forward returns one output per parameter
[PASS] TODO 5: terrain head outputs (8, 2)
[PASS] TODO 5: city_archetype head outputs (8, 3)
[PASS] TODO 5: num_clusters head outputs (8, 1)
[PASS] TODO 5: density head outputs (8, 1)
[PASS] TODO 5: cluster_tightness head outputs (8, 1)
[PASS] TODO 5: relief head outputs (8, 1)
[PASS] TODO 5: water_level head outputs (8, 1)
[PASS] TODO 5: green_amount head outputs (8, 1)
[PASS] TODO 5: green_override head outputs (8, 1)
[PASS] TODO 5: road_continuity head outputs (8, 1)

trainable weights (hidden_dim=64): 29,645


---

## Part 5 — Training: `train.py`

Each kind of head needs its own loss:

| kind | loss | in plain words |
|---|---|---|
| categorical | cross-entropy | push up the score of the right choice |
| continuous | mean squared error | how far off the number is, squared |
| binary | binary cross-entropy | how confident it was in the wrong answer |

The total loss is just their sum. Then the standard PyTorch training step:
forward, loss, `zero_grad`, `backward`, `step`.

Given: class weights (read the docstring: why "monocentric" would win every
argument otherwise), metrics, saving. You write **TODO 7–8**.

In [41]:
%%writefile text2city/train.py
#!/usr/bin/env python3
"""
Train the text-to-city-parameter model.

    python text2city/train.py                       # sentence-transformer (default)
    python text2city/train.py --embedder tfidf      # offline fallback

Saves text2city/checkpoint/model.pt (trunk + heads) and the embedder file,
so evaluate.py can load everything from one place.
"""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from text2city.schema import PARAM_SCHEMA  # noqa: E402
from text2city.dataset import load_rows, build_targets, train_val_split  # noqa: E402
from text2city.embedders import get_embedder, embedder_filename  # noqa: E402
from text2city.model import TextToCityParams  # noqa: E402
from text2city.freshness import fingerprint_sources  # noqa: E402


def compute_class_weights(targets: dict, train_idx) -> dict:
    """
    Inverse-frequency weights for each categorical parameter.

    A categorical default ("monocentric", "hills") is ALSO one of its real
    choices, so every sentence that doesn't mention it adds to that class.
    Left alone, the model learns "when unsure, say monocentric". Weighting
    rare classes up removes that shortcut.
    """
    weights = {}
    for spec in PARAM_SCHEMA:
        if spec.kind != "categorical":
            continue
        t = targets[spec.name][train_idx]
        counts = np.array([np.sum(t == i) for i in range(len(spec.choices))], dtype=np.float32)
        inv = 1.0 / np.maximum(counts, 1)
        weights[spec.name] = torch.tensor(inv / inv.sum() * len(spec.choices), dtype=torch.float32)
    return weights


def compute_loss(raw_outputs: dict, targets: dict, class_weights: dict) -> torch.Tensor:
    """
    Sum of one loss per parameter, each picked to match the head's kind:

        categorical -> cross-entropy on the logits (with class weights)
        continuous  -> mean squared error between sigmoid(output) and the 0-1 target
        binary      -> binary cross-entropy on the logit

    raw_outputs[name] has shape (batch, k); continuous and binary heads have
    k=1, so .squeeze(-1) them to (batch,) to line up with their targets.
    """
    total = torch.tensor(0.0)
    for spec in PARAM_SCHEMA:
        pred = raw_outputs[spec.name]
        target = torch.as_tensor(targets[spec.name])
        # TODO 7: pick the loss that matches this head's kind (docstring above).
        # All three live in nn.functional:
        #   cross_entropy(logits, class_indices, weight=class_weights[spec.name])
        #   mse_loss(prediction, target)      <- squash pred with torch.sigmoid first
        #   binary_cross_entropy_with_logits(logit, target)
        if spec.kind == "categorical":
            loss = nn.functional.cross_entropy(pred, target, weight=class_weights[spec.name])
        elif spec.kind == "continuous":
            loss = nn.functional.mse_loss(torch.sigmoid(pred.squeeze(-1)), target)
        else:  # binary
            loss = nn.functional.binary_cross_entropy_with_logits(pred.squeeze(-1), target)
        total = total + loss
    return total


@torch.no_grad()
def evaluate_split(model, embeddings, targets, idx) -> dict:
    """Accuracy for categorical/binary heads, MAE in real units for continuous ones."""
    model.eval()
    raw = model(torch.as_tensor(embeddings[idx]))
    metrics = {}
    for spec in PARAM_SCHEMA:
        target = targets[spec.name][idx]
        if spec.kind == "categorical":
            pred = torch.argmax(raw[spec.name], dim=1).numpy()
            metrics[spec.name] = float((pred == target).mean())
        elif spec.kind == "continuous":
            low, high = spec.value_range
            pred = low + torch.sigmoid(raw[spec.name].squeeze(-1)).numpy() * (high - low)
            metrics[spec.name] = float(np.mean(np.abs(pred - targets[f"{spec.name}__raw"][idx])))
        else:
            pred = (torch.sigmoid(raw[spec.name].squeeze(-1)) > 0.5).numpy().astype(np.float32)
            metrics[spec.name] = float((pred == target).mean())
    model.train()
    return metrics


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset", type=str, default="data/text2city_dataset.csv")
    parser.add_argument("--embedder", choices=["sentence-transformer", "tfidf"], default="sentence-transformer")
    parser.add_argument("--epochs", type=int, default=60)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--hidden-dim", type=int, default=192)
    parser.add_argument("--val-fraction", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--out", type=str, default="text2city/checkpoint")
    args = parser.parse_args()

    torch.manual_seed(args.seed)

    rows = load_rows(args.dataset)
    texts = [r["text"] for r in rows]
    targets = build_targets(rows)
    train_idx, val_idx = train_val_split(len(rows), args.val_fraction, seed=args.seed)
    print(f"{len(rows)} examples ({len(train_idx)} train / {len(val_idx)} val)")

    class_weights = compute_class_weights(targets, train_idx)

    print(f"embedding sentences with '{args.embedder}' ...")
    t0 = time.time()
    embedder = get_embedder(args.embedder)
    embedder.fit([texts[i] for i in train_idx])
    embeddings = embedder.encode(texts)
    print(f"  {embeddings.shape[0]} sentences -> dim {embeddings.shape[1]} in {time.time() - t0:.1f}s")

    model = TextToCityParams(embedding_dim=embeddings.shape[1], hidden_dim=args.hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    print(f"\ntraining for {args.epochs} epochs ...")
    t0 = time.time()
    rng = np.random.default_rng(args.seed)
    for epoch in range(args.epochs):
        perm = rng.permutation(train_idx)
        epoch_loss = 0.0
        for start in range(0, len(perm), args.batch_size):
            batch_idx = perm[start:start + args.batch_size]
            batch_emb = torch.as_tensor(embeddings[batch_idx])
            batch_targets = {k: v[batch_idx] for k, v in targets.items()}

            # TODO 8: one optimisation step, in this order:
            #   1. forward pass: the model's raw outputs for batch_emb
            #   2. the loss, from compute_loss(raw, batch_targets, class_weights)
            #   3. clear the previous step's gradients   optimizer.zero_grad()
            #   4. backpropagate                          loss.backward()
            #   5. update the weights                     optimizer.step()
            raw = model(batch_emb)
            loss = compute_loss(raw, batch_targets, class_weights)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(batch_idx)

        if epoch % 10 == 0 or epoch == args.epochs - 1:
            m = evaluate_split(model, embeddings, targets, val_idx)
            print(f"  epoch {epoch:3d}  train_loss={epoch_loss / len(train_idx):.4f}  "
                  f"val terrain_acc={m['terrain']:.3f}  city_acc={m['city_archetype']:.3f}  "
                  f"density_MAE={m['density']:.3f}")

    print(f"\ntraining done in {time.time() - t0:.1f}s")
    print("\nvalidation (accuracy for categorical/binary, MAE in real units for continuous):")
    final = evaluate_split(model, embeddings, targets, val_idx)
    for spec in PARAM_SCHEMA:
        label = "acc" if spec.kind in ("categorical", "binary") else "MAE"
        print(f"  {spec.name:20s} {label}={final[spec.name]:.4f}")

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state": model.state_dict(),
        "embedding_dim": embeddings.shape[1],
        "hidden_dim": args.hidden_dim,
        "embedder_kind": args.embedder,
        "fingerprint": fingerprint_sources(args.dataset),
    }, out_dir / "model.pt")
    embedder.save(str(out_dir / embedder_filename(args.embedder)))
    print(f"\nsaved checkpoint to {out_dir}/")


if __name__ == "__main__":
    main()


Overwriting text2city/train.py


In [42]:
tr = fresh("text2city.train")
if tr and mm:
    try:
        torch.manual_seed(0)
        model = tr.TextToCityParams(embedding_dim=384, hidden_dim=64)
        x = torch.randn(32, 384)
        fake = {}
        for p in tr.PARAM_SCHEMA:
            if p.kind == "categorical":
                fake[p.name] = np.random.randint(0, len(p.choices), size=32).astype(np.int64)
            else:
                fake[p.name] = np.random.rand(32).astype(np.float32)
        fake["green_override"] = (fake["green_override"] > 0.5).astype(np.float32)
        weights = {p.name: torch.ones(len(p.choices)) for p in tr.PARAM_SCHEMA if p.kind == "categorical"}

        loss = tr.compute_loss(model(x), fake, weights)
        ok = isinstance(loss, torch.Tensor) and loss.dim() == 0 and torch.isfinite(loss) and loss.item() > 0
        check("TODO 7: compute_loss returns one positive number", ok,
              f"got {loss!r}" if not isinstance(loss, torch.Tensor) else f"shape {tuple(loss.shape)}")

        opt = torch.optim.Adam(model.parameters(), lr=1e-2)
        first = loss.item()
        for _ in range(50):
            l = tr.compute_loss(model(x), fake, weights)
            opt.zero_grad(); l.backward(); opt.step()
        check("TODO 7: the loss goes down when you train on it", l.item() < first * 0.7,
              f"{first:.3f} -> {l.item():.3f}. A loss the model can't reduce usually means the wrong "
              "loss for a kind, or a missing torch.sigmoid on the continuous head.")
    except Exception as e:
        check("TODO 7 runs", False, f"{type(e).__name__}: {e}")

[PASS] TODO 7: compute_loss returns one positive number
[PASS] TODO 7: the loss goes down when you train on it


Now train for real. About a minute: most of it is embedding 15,000 sentences once, then 60 short epochs. The `train_loss` should fall steadily.

In [43]:
# Quick preview run now. For the FULL run later, set EPOCHS = 60 and re-run this cell + the evaluate cell.
EPOCHS = 20
ok, out = run([sys.executable, "text2city/train.py", "--embedder", EMBEDDER, "--epochs", EPOCHS])

losses = [float(line.split("train_loss=")[1].split()[0]) for line in out.splitlines() if "train_loss=" in line]
check("TODO 8: training ran to the end", ok and "saved checkpoint" in out, "Read the traceback above.")
check("TODO 8: train_loss went down", len(losses) >= 2 and losses[-1] < losses[0] * 0.5,
      f"losses: {losses}. Flat loss = optimizer.step() never runs, or zero_grad/backward are missing.")
check("checkpoint saved", (REPO / "text2city" / "checkpoint" / "model.pt").exists())

$ /usr/local/bin/python3 text2city/train.py --embedder sentence-transformer --epochs 20 

15000 examples (12750 train / 2250 val)
embedding sentences with 'sentence-transformer' ...
W1101 13:02:13.411000 25273 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16165.14it/s]
  15000 sentences -> dim 384 in 10.6s

training for 20 epochs ...
  epoch   0  train_loss=1.6411  val terrain_acc=0.979  city_acc=0.893  density_MAE=0.159
  epoch  10  train_loss=0.2057  val terrain_acc=1.000  city_acc=0.957  density_MAE=0.104
  epoch  19  train_loss=0.1490  val terrain_acc=1.000  city_acc=0.960  density_MAE=0.091

training done in 3.6s

validation (accuracy for categorical/binary, MAE in real units for continuous):
  terrain              acc=1.0000
  city_archetype       acc=0.9604
  num_clusters         MAE=0.2036
  density              MAE=0.0908
  cluster_tightness

True

---

## Part 6 — Evaluation: `evaluate.py`

Two tests, on purpose:

1. **Held-out synthetic sentences.** Same phrase banks, examples it never
   trained on. Expect near-perfect scores.
2. **Hand-written sentences.** Phrased like a person types. **This is the
   one that matters.** Near-perfect on (1) and poor on (2) means the model
   memorised the templates instead of learning the task.

You write **TODO 9** (the metrics) and **TODO 10** (your own test
sentences: try to catch the model out).

In [46]:
%%writefile text2city/evaluate.py
#!/usr/bin/env python3
"""
Evaluate a trained checkpoint, two ways -- deliberately kept separate:

  1. Held-out synthetic sentences: same phrase banks as training, but
     examples the model never trained on. Tells you if it learned the
     mapping at all.
  2. Hand-written sentences no template could have produced: tells you if
     it generalises to how a person actually types. This is the number
     that matters. A model that aces (1) and fails (2) memorised the
     templates instead of learning the task.

    python text2city/evaluate.py
"""
import argparse
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from text2city.schema import PARAM_SCHEMA, PARAM_BY_NAME  # noqa: E402
from text2city.dataset import load_rows, build_targets, train_val_split  # noqa: E402
from text2city.embedders import load_embedder, embedder_filename  # noqa: E402
from text2city.model import TextToCityParams, decode_batch  # noqa: E402
from text2city.freshness import check_staleness  # noqa: E402


# Hand-written sentences, each with what a person would EXPECT the model to
# get right. Only list the parameters the sentence actually talks about.
#   categorical -> the exact choice, e.g. "polycentric"
#   continuous  -> "high" or "low" (relative to that parameter's range),
#                  or an int for num_clusters (within 0.75 counts)
#   binary      -> True / False
HANDWRITTEN_EXAMPLES = [
    ("a huge sprawling metropolis packed with skyscrapers", {"density": "high"}),
    ("a quiet little town on flat land with hardly any buildings", {"terrain": "flat", "density": "low"}),
    ("a city in the hills with lots of parks everywhere", {"terrain": "hills", "green_amount": "high"}),
    ("a city with three separate business hubs", {"city_archetype": "polycentric", "num_clusters": 3}),
    ("a long thin city stretched along one road", {"city_archetype": "linear"}),
    ("a sparse town with almost no roads", {"density": "low", "road_continuity": "low"}),
    # TODO 10: add at least THREE sentences of your own below, phrased the way
    # a person would actually type -- not copied from the phrase banks in
    # generate_dataset.py. Try to catch the model out.
    ("a tiny village in the middle of nowhere, mostly fields and trees, hardly any streets",
     {"density": "low", "green_amount": "high", "road_continuity": "low"}),
    ("flat as a pancake and totally built over, skyscrapers everywhere", {"terrain": "flat", "density": "high"}),
    ("a pair of busy downtown areas on rolling hills", {"terrain": "hills", "city_archetype": "polycentric"}),
    ("a crowded, gridlocked metropolis with a well connected street grid and almost no parks",
     {"density": "high", "road_continuity": "high", "green_amount": "low"}),
]


def expectation_met(spec, predicted, expected) -> bool:
    if spec.kind in ("categorical", "binary"):
        return predicted == expected
    low, high = spec.value_range
    if expected == "high":
        return predicted > low + 0.6 * (high - low)
    if expected == "low":
        return predicted < low + 0.4 * (high - low)
    return abs(predicted - float(expected)) <= 0.75


def load_checkpoint(checkpoint_dir: str, dataset_path: str):
    ckpt_dir = Path(checkpoint_dir)
    ckpt = torch.load(ckpt_dir / "model.pt", map_location="cpu", weights_only=False)
    check_staleness(ckpt.get("fingerprint"), dataset_path)
    model = TextToCityParams(embedding_dim=ckpt["embedding_dim"], hidden_dim=ckpt["hidden_dim"])
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    kind = ckpt["embedder_kind"]
    embedder = load_embedder(kind, str(ckpt_dir / embedder_filename(kind)))
    return model, embedder


def predict(model, embedder, texts: list[str]) -> list[dict]:
    with torch.no_grad():
        raw = model(torch.as_tensor(embedder.encode(texts)))
    return [decode_batch(raw, i) for i in range(len(texts))]


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=str, default="text2city/checkpoint")
    parser.add_argument("--dataset", type=str, default="data/text2city_dataset.csv")
    parser.add_argument("--val-fraction", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    model, embedder = load_checkpoint(args.checkpoint, args.dataset)

    # ---- 1. held-out synthetic examples ----
    rows = load_rows(args.dataset)
    targets = build_targets(rows)
    _, val_idx = train_val_split(len(rows), args.val_fraction, seed=args.seed)
    preds = predict(model, embedder, [rows[i]["text"] for i in val_idx])

    print("=== 1. held-out synthetic examples ===")
    for spec in PARAM_SCHEMA:
        if spec.kind in ("categorical", "binary"):
            # TODO 9a: accuracy = the fraction of validation rows where the
            # prediction matches the label. zip(val_idx, preds) pairs each row
            # index i with its prediction p. The CSV stores everything as text,
            # so compare str(p[spec.name]) with rows[i][spec.name].
            # np.mean of a list of True/False gives the fraction that are True.
            accuracy = np.mean([str(p[spec.name]) == rows[i][spec.name] for i, p in zip(val_idx, preds)])
            print(f"  {spec.name:20s} accuracy = {accuracy:.3f}")
        else:
            # mean absolute error in the parameter's real units
            predicted = np.array([p[spec.name] for p in preds])
            actual = targets[f"{spec.name}__raw"][val_idx]
            # TODO 9b: mean absolute error between predicted and actual.
            mae = np.mean(np.abs(predicted - actual))
            print(f"  {spec.name:20s} MAE      = {mae:.4f}")

    # ---- 2. hand-written sentences ----
    print("\n=== 2. hand-written sentences (the generalisation test) ===")
    hw_preds = predict(model, embedder, [text for text, _ in HANDWRITTEN_EXAMPLES])
    hits = total = 0
    for (text, expected), pred in zip(HANDWRITTEN_EXAMPLES, hw_preds):
        print(f'\n  "{text}"')
        for name, want in expected.items():
            if name not in PARAM_BY_NAME:
                print(f"      {name:20s} (not a parameter -- check the spelling)")
                continue
            if name == "terrain" and want not in PARAM_BY_NAME["terrain"].choices:
                print(f"      {name:20s} skipped: your citygen has no {want!r} terrain")
                continue
            got = pred[name]
            ok = expectation_met(PARAM_BY_NAME[name], got, want)
            hits += ok
            total += 1
            shown = round(got, 2) if isinstance(got, float) else got
            print(f"      {'PASS' if ok else 'MISS'}  {name:20s} wanted {want!r:14} got {shown!r}")
    if total:
        print(f"\nhand-written expectations met: {hits}/{total} ({hits / total:.0%})")


if __name__ == "__main__":
    main()


Overwriting text2city/evaluate.py


In [47]:
ev = fresh("text2city.evaluate")
if ev:
    n = len(ev.HANDWRITTEN_EXAMPLES)
    check("TODO 10: at least 3 sentences of your own (9+ in total)", n >= 9, f"found {n}")
    bad = [name for _, exp in ev.HANDWRITTEN_EXAMPLES for name in exp if name not in ev.PARAM_BY_NAME]
    check("TODO 10: every expected key is a real parameter name", not bad, f"unknown: {bad}")

ok, out = run([sys.executable, "text2city/evaluate.py"])
check("TODO 9: evaluate.py runs", ok and "accuracy =" in out and "MAE" in out, "Read the traceback above.")
accs = [float(l.split("=")[1]) for l in out.splitlines() if "accuracy =" in l]
check("TODO 9: held-out accuracies look right (above 0.9)", bool(accs) and min(accs) > 0.9,
      f"accuracies: {accs}. Below 0.5 everywhere? Check the str() comparison in TODO 9a.")

[PASS] TODO 10: at least 3 sentences of your own (9+ in total)
[PASS] TODO 10: every expected key is a real parameter name
$ /usr/local/bin/python3 text2city/evaluate.py 

W1101 13:07:35.475000 25380 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9593.05it/s]
=== 1. held-out synthetic examples ===
  terrain              accuracy = 1.000
  city_archetype       accuracy = 0.960
  num_clusters         MAE      = 0.2036
  density              MAE      = 0.0908
  cluster_tightness    MAE      = 0.0707
  relief               MAE      = 0.1088
  water_level          MAE      = 0.0396
  green_amount         MAE      = 0.0690
  green_override       accuracy = 1.000
  road_continuity      MAE      = 0.0551

=== 2. hand-written sentences (the generalisation test) ===

  "a huge sprawling metropolis packed with skyscrapers"
      PASS  density              wanted

True

---

## Part 7 — What did you find?

Before you leave, answer these with the people next to you:

1. How big is the gap between test 1 and test 2? Why does it exist?
2. Which of your own sentences did it miss? Is that a **vocabulary** problem
   (the phrase banks never use that word) or a **default** problem (it fell
   back to the default because nothing in the sentence matched)?
3. `"a quiet little town … with hardly any buildings"` usually misses on
   density. Why might "hardly any buildings" not push density down?

### Try one experiment

Pick one, re-run `generate_dataset.py` → `train.py` → `evaluate.py`, and
compare the hand-written score:

- Add a phrase to a bank in `generate_dataset.py` that fixes one of your misses.
- Lower `--epochs` to 5. Does held-out drop? Does hand-written?
- Train with `--embedder tfidf` and compare. Which sentences does it lose, and why?

If you edit a file but forget to retrain, `evaluate.py` prints a **STALE**
warning. That's on purpose.

## Checklist

- [ ] All five files written, every check `[PASS]`
- [ ] `data/text2city_dataset.csv` (15,000 rows) and `text2city/checkpoint/` exist
- [ ] `evaluate.py` runs, with at least 3 sentences of your own
- [ ] Committed and pushed

**Next time (Week 7):** `infer.py` — type a sentence, get a city in Blender.